# NMO_projekt

Notebook zawiera kompletny skrypt do optymalizacji portfela Markowitza z ograniczeniem kardynalności. Opisy są ograniczone do informacji o działaniu kolejnych bloków kodu.

## 1. Importy i ustawienia

Ładowane są biblioteki, ustawiane ziarno losowe, liczba dni handlowych oraz podstawowe ustawienia wykresów.

In [ ]:
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 2026
TRADING_DAYS = 252
np.set_printoptions(precision=6, suppress=True)
plt.rcParams["axes.unicode_minus"] = False

## 2. Wczytanie danych i klasy aktywów

Kod szuka pliku `synthetic_returns.csv`, wczytuje stopy zwrotu, oblicza średnie, macierz kowariancji i dzieli aktywa na klasy według empirycznej zmienności.

In [ ]:
def find_returns_file() -> Path | None:
    cwd = Path.cwd()
    candidates = [cwd / "data" / "synthetic_returns.csv", cwd / "synthetic_returns.csv"]
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / "Data" / "synthetic_returns.csv")
    return next((path for path in candidates if path.exists()), None)


def generate_synthetic_returns(T=1000, N=30, seed=RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    market = rng.normal(0.00025, 0.008, T)
    loadings = rng.uniform(0.4, 1.3, N)
    idio_vol = np.r_[
        rng.uniform(0.003, 0.007, N // 3),
        rng.uniform(0.007, 0.012, N // 3),
        rng.uniform(0.012, 0.022, N - 2 * (N // 3)),
    ]
    means = rng.uniform(0.00005, 0.00065, N)
    eps = rng.standard_t(df=6, size=(T, N)) * idio_vol
    data = means + market[:, None] * loadings + eps
    return pd.DataFrame(data, columns=[f"Asset_{i}" for i in range(N)])

returns_path = find_returns_file()
if returns_path is None:
    returns_df = generate_synthetic_returns()
    print("Nie znaleziono pliku CSV. Wygenerowano dane syntetyczne.")
else:
    returns_df = pd.read_csv(returns_path)
    print(f"Wczytano dane z: {returns_path}")

returns_df = returns_df.apply(pd.to_numeric, errors="coerce").dropna(axis=0, how="any")
returns = returns_df.to_numpy(dtype=float)
mu = returns.mean(axis=0)
cov = np.cov(returns, rowvar=False)
asset_names = returns_df.columns.to_list()
n_assets = returns.shape[1]

asset_vol = np.sqrt(np.diag(cov))
q1, q2 = np.quantile(asset_vol, [1/3, 2/3])
asset_class = np.where(asset_vol <= q1, "konserwatywne", np.where(asset_vol <= q2, "zrównoważone", "agresywne"))

asset_summary = pd.DataFrame({
    "asset": asset_names,
    "class": asset_class,
    "mean_daily_return": mu,
    "annualized_mean_return": mu * TRADING_DAYS,
    "daily_volatility": asset_vol,
    "annualized_volatility": asset_vol * np.sqrt(TRADING_DAYS),
})

print(f"Liczba obserwacji: {returns.shape[0]}")
print(f"Liczba aktywów: {n_assets}")
asset_summary.head(10)

In [ ]:
class_summary = (
    asset_summary
    .groupby("class")
    .agg(
        assets=("asset", "count"),
        mean_annual_return=("annualized_mean_return", "mean"),
        mean_annual_volatility=("annualized_volatility", "mean"),
    )
    .reindex(["konserwatywne", "zrównoważone", "agresywne"])
)
class_summary

## 3. Funkcje portfelowe i ograniczenia

Funkcje normalizują i naprawiają wagi, liczą zwrot, wariancję, zmienność, dywersyfikację, udziały klas aktywów oraz sprawdzają ograniczenia portfela.

In [ ]:
def normalize_weights(weights: np.ndarray) -> np.ndarray:
    w = np.asarray(weights, dtype=float)
    total = w.sum()
    if not np.isfinite(total) or total <= 0:
        return np.ones_like(w) / len(w)
    return w / total


def repair_weights(weights: np.ndarray, max_assets: int) -> np.ndarray:
    w = np.asarray(weights, dtype=float).copy()
    w[~np.isfinite(w)] = 0.0
    w = np.clip(w, 0.0, None)
    if max_assets < len(w):
        keep = np.argsort(w)[-max_assets:]
        mask = np.zeros_like(w, dtype=bool)
        mask[keep] = True
        w[~mask] = 0.0
    if w.sum() <= 0:
        keep = np.arange(min(max_assets, len(w)))
        w[keep] = 1.0
    return normalize_weights(w)


def cardinality(weights: np.ndarray, threshold: float = 1e-8) -> int:
    return int(np.sum(np.asarray(weights) > threshold))


def portfolio_return(weights: np.ndarray, mean_returns: np.ndarray) -> float:
    return float(np.dot(weights, mean_returns))


def portfolio_variance(weights: np.ndarray, cov_matrix: np.ndarray) -> float:
    return float(weights @ cov_matrix @ weights)


def portfolio_volatility(weights: np.ndarray, cov_matrix: np.ndarray) -> float:
    return float(np.sqrt(max(portfolio_variance(weights, cov_matrix), 0.0)))


def diversification_metrics(weights: np.ndarray) -> dict:
    w = np.asarray(weights, dtype=float)
    active_w = w[w > 1e-8]
    hhi = float(np.sum(w**2))
    effective_n = float(1.0 / hhi) if hhi > 0 else 0.0
    entropy = float(-np.sum(active_w * np.log(active_w))) if len(active_w) else 0.0
    max_entropy = np.log(len(active_w)) if len(active_w) > 1 else 1.0
    return {
        "hhi": hhi,
        "effective_assets": effective_n,
        "entropy_norm": entropy / max_entropy if max_entropy > 0 else 0.0,
        "max_weight": float(w.max()),
        "top3_weight": float(np.sort(w)[-3:].sum()),
    }


def class_weights(weights: np.ndarray) -> dict:
    out = {}
    for cls in ["konserwatywne", "zrównoważone", "agresywne"]:
        out[f"share_{cls}"] = float(weights[asset_class == cls].sum())
    return out


def enforce_target_return(weights: np.ndarray, mean_returns: np.ndarray, max_assets: int, target_return: float) -> np.ndarray:
    w = repair_weights(weights, max_assets)
    if portfolio_return(w, mean_returns) + 1e-10 >= target_return:
        return w
    best_asset = int(np.argmax(mean_returns))
    active = np.where(w > 1e-8)[0]
    if w[best_asset] <= 1e-8 and len(active) >= max_assets:
        drop_candidates = [idx for idx in active if idx != best_asset]
        if drop_candidates:
            drop = min(drop_candidates, key=lambda idx: w[idx])
            w[drop] = 0.0
            w = normalize_weights(w)
    current_return = portfolio_return(w, mean_returns)
    best_return = float(mean_returns[best_asset])
    if best_return <= current_return:
        return repair_weights(w, max_assets)
    lam = np.clip((target_return - current_return) / (best_return - current_return) + 1e-6, 0.0, 1.0)
    w = (1.0 - lam) * w
    w[best_asset] += lam
    return repair_weights(w, max_assets)


def is_feasible(weights: np.ndarray, max_assets: int, target_return: float, tolerance: float = 1e-8) -> bool:
    w = np.asarray(weights)
    return (
        np.all(w >= -tolerance)
        and abs(w.sum() - 1.0) <= 1e-6
        and cardinality(w, tolerance) <= max_assets
        and portfolio_return(w, mu) + tolerance >= target_return
    )


def make_objective(mean_returns: np.ndarray, cov_matrix: np.ndarray, max_assets: int, target_return: float, penalty: float = 1e4):
    def objective(raw_weights: np.ndarray) -> float:
        w = enforce_target_return(raw_weights, mean_returns, max_assets, target_return)
        variance = portfolio_variance(w, cov_matrix)
        shortfall = max(0.0, target_return - portfolio_return(w, mean_returns))
        return variance + penalty * shortfall**2
    return objective


def portfolio_metrics(weights: np.ndarray, max_assets: int, target_return: float, objective_value: float, elapsed: float, algorithm: str, run: int) -> dict:
    daily_return = portfolio_return(weights, mu)
    daily_variance = portfolio_variance(weights, cov)
    daily_vol = np.sqrt(max(daily_variance, 0.0))
    row = {
        "algorithm": algorithm,
        "run": run,
        "A": max_assets,
        "objective": objective_value,
        "daily_return": daily_return,
        "annual_return": daily_return * TRADING_DAYS,
        "daily_variance": daily_variance,
        "annual_variance": daily_variance * TRADING_DAYS,
        "daily_volatility": daily_vol,
        "annual_volatility": daily_vol * np.sqrt(TRADING_DAYS),
        "cardinality": cardinality(weights),
        "feasible": is_feasible(weights, max_assets, target_return),
        "time_sec": elapsed,
        "weights": weights,
    }
    row.update(diversification_metrics(weights))
    row.update(class_weights(weights))
    return row

## 4. Parametry eksperymentu

W tej sekcji ustawiane są wartości `A`, próg minimalnego zwrotu `r0`, liczba powtórzeń i parametry iteracyjne algorytmów.

In [ ]:
A_VALUES = [2, 3, 5, 10, 15]
TARGET_RETURN = float(np.median(mu))
N_RUNS = 3

SA_ITERATIONS = 450
GA_GENERATIONS = 80
GA_POPULATION = 45
PSO_ITERATIONS = 90
PSO_PARTICLES = 45

print(f"Minimalna dzienna oczekiwana stopa zwrotu r0: {TARGET_RETURN:.8f}")
print(f"Minimalna roczna oczekiwana stopa zwrotu r0: {TARGET_RETURN * TRADING_DAYS:.4%}")

## 5. Algorytmy optymalizacyjne

Zdefiniowane są trzy metody rozwiązujące ten sam problem minimalizacji wariancji: SA, GA i PSO.

In [ ]:
def simulated_annealing(objective, n_dim: int, max_assets: int, seed: int, n_iter: int = SA_ITERATIONS, t0: float = 1.0, alpha: float = 0.995, step: float = 0.12):
    rng = np.random.default_rng(seed)
    current = enforce_target_return(rng.random(n_dim), mu, max_assets, TARGET_RETURN)
    current_score = objective(current)
    best = current.copy()
    best_score = current_score
    history = [best_score]
    accepted = 0
    for k in range(n_iter):
        temp = t0 * (alpha ** k)
        candidate = current + rng.normal(0.0, step * max(temp, 0.05), n_dim)
        candidate = enforce_target_return(candidate, mu, max_assets, TARGET_RETURN)
        candidate_score = objective(candidate)
        delta = candidate_score - current_score
        if delta <= 0 or rng.random() < np.exp(-delta / max(temp, 1e-12)):
            current, current_score = candidate, candidate_score
            accepted += 1
        if current_score < best_score:
            best, best_score = current.copy(), current_score
        history.append(best_score)
    return {"weights": best, "objective": best_score, "history": history, "accepted": accepted}


def tournament_select(population: np.ndarray, scores: np.ndarray, rng: np.random.Generator, k: int = 3) -> np.ndarray:
    idx = rng.integers(0, len(population), size=k)
    return population[idx[np.argmin(scores[idx])]].copy()


def genetic_algorithm(objective, n_dim: int, max_assets: int, seed: int, population_size: int = GA_POPULATION, generations: int = GA_GENERATIONS, mutation_rate: float = 0.12, elite_frac: float = 0.08):
    rng = np.random.default_rng(seed)
    population = np.array([enforce_target_return(rng.random(n_dim), mu, max_assets, TARGET_RETURN) for _ in range(population_size)])
    scores = np.array([objective(ind) for ind in population])
    elite_count = max(1, int(population_size * elite_frac))
    history = [float(scores.min())]
    for _ in range(generations):
        elite_idx = np.argsort(scores)[:elite_count]
        new_population = [population[i].copy() for i in elite_idx]
        while len(new_population) < population_size:
            p1 = tournament_select(population, scores, rng)
            p2 = tournament_select(population, scores, rng)
            child = np.where(rng.random(n_dim) < 0.5, p1, p2)
            if rng.random() < mutation_rate:
                child += rng.normal(0.0, 0.08, n_dim)
            new_population.append(enforce_target_return(child, mu, max_assets, TARGET_RETURN))
        population = np.array(new_population)
        scores = np.array([objective(ind) for ind in population])
        history.append(float(scores.min()))
    best_idx = int(np.argmin(scores))
    return {"weights": population[best_idx].copy(), "objective": float(scores[best_idx]), "history": history}


def particle_swarm_optimization(objective, n_dim: int, max_assets: int, seed: int, n_particles: int = PSO_PARTICLES, n_iter: int = PSO_ITERATIONS, inertia: float = 0.72, cognitive: float = 1.45, social: float = 1.45):
    rng = np.random.default_rng(seed)
    positions = np.array([enforce_target_return(rng.random(n_dim), mu, max_assets, TARGET_RETURN) for _ in range(n_particles)])
    velocities = rng.normal(0.0, 0.04, size=(n_particles, n_dim))
    personal_best = positions.copy()
    personal_scores = np.array([objective(p) for p in positions])
    global_idx = int(np.argmin(personal_scores))
    global_best = personal_best[global_idx].copy()
    global_score = float(personal_scores[global_idx])
    history = [global_score]
    for _ in range(n_iter):
        r1 = rng.random((n_particles, n_dim))
        r2 = rng.random((n_particles, n_dim))
        velocities = inertia * velocities + cognitive * r1 * (personal_best - positions) + social * r2 * (global_best - positions)
        velocities = np.clip(velocities, -0.25, 0.25)
        positions = np.array([enforce_target_return(pos + vel, mu, max_assets, TARGET_RETURN) for pos, vel in zip(positions, velocities)])
        scores = np.array([objective(p) for p in positions])
        improved = scores < personal_scores
        personal_best[improved] = positions[improved]
        personal_scores[improved] = scores[improved]
        candidate_idx = int(np.argmin(personal_scores))
        if personal_scores[candidate_idx] < global_score:
            global_best = personal_best[candidate_idx].copy()
            global_score = float(personal_scores[candidate_idx])
        history.append(global_score)
    return {"weights": global_best, "objective": global_score, "history": history}

## 6. Uruchomienie eksperymentu

Pętla eksperymentalna uruchamia każdy algorytm dla każdej wartości `A` i zapisuje wyniki oraz historie zbieżności.

In [ ]:
@dataclass
class AlgorithmSpec:
    name: str
    runner: callable

algorithms = [
    AlgorithmSpec("SA", simulated_annealing),
    AlgorithmSpec("GA", genetic_algorithm),
    AlgorithmSpec("PSO", particle_swarm_optimization),
]

all_results = []
histories = {}

for max_assets in A_VALUES:
    objective = make_objective(mu, cov, max_assets=max_assets, target_return=TARGET_RETURN)
    for algorithm in algorithms:
        for run in range(N_RUNS):
            seed = RANDOM_SEED + 1000 * max_assets + 100 * run + len(algorithm.name)
            start = time.perf_counter()
            result = algorithm.runner(objective, n_assets, max_assets, seed)
            elapsed = time.perf_counter() - start
            weights = enforce_target_return(result["weights"], mu, max_assets, TARGET_RETURN)
            all_results.append(portfolio_metrics(weights, max_assets, TARGET_RETURN, result["objective"], elapsed, algorithm.name, run))
            histories[(algorithm.name, max_assets, run)] = result["history"]

results = pd.DataFrame(all_results)
results_display = results.drop(columns=["weights"])
results_display.head(12)

## 7. Tabela podsumowująca

Wyniki są agregowane po algorytmie i wartości `A`; tabela zawiera metryki jakości, stabilności, czasu, dywersyfikacji i udziałów klas aktywów.

In [ ]:
summary_results = (
    results
    .groupby(["A", "algorithm"])
    .agg(
        mean_objective=("objective", "mean"),
        std_objective=("objective", "std"),
        best_objective=("objective", "min"),
        worst_objective=("objective", "max"),
        mean_daily_variance=("daily_variance", "mean"),
        best_daily_variance=("daily_variance", "min"),
        mean_annual_volatility=("annual_volatility", "mean"),
        mean_annual_return=("annual_return", "mean"),
        feasible_rate=("feasible", "mean"),
        mean_effective_assets=("effective_assets", "mean"),
        mean_hhi=("hhi", "mean"),
        mean_max_weight=("max_weight", "mean"),
        mean_top3_weight=("top3_weight", "mean"),
        mean_share_conservative=("share_konserwatywne", "mean"),
        mean_share_balanced=("share_zrównoważone", "mean"),
        mean_share_aggressive=("share_agresywne", "mean"),
        mean_time_sec=("time_sec", "mean"),
    )
    .reset_index()
    .sort_values(["A", "best_daily_variance"])
)
summary_results

## 8. Najlepszy portfel

Wybierany jest portfel o najniższej wariancji spośród wszystkich dopuszczalnych wyników.

In [ ]:
best_idx = results.groupby(["A", "algorithm"])["daily_variance"].idxmin()
best_by_algorithm_a = results.loc[best_idx].copy().sort_values(["A", "daily_variance"])
best_overall = results.loc[int(results["daily_variance"].idxmin())]

print("Najlepszy wynik według minimalnej wariancji:")
print(f"Algorytm: {best_overall['algorithm']}")
print(f"A: {best_overall['A']}")
print(f"Dzienna wariancja: {best_overall['daily_variance']:.10f}")
print(f"Roczna oczekiwana stopa zwrotu: {best_overall['annual_return']:.4%}")
print(f"Roczna zmienność: {best_overall['annual_volatility']:.4%}")
print(f"Efektywna liczba aktywów: {best_overall['effective_assets']:.2f}")
print(f"HHI: {best_overall['hhi']:.4f}")
print(f"Największa pojedyncza waga: {best_overall['max_weight']:.2%}")
print(f"Spełnia ograniczenia: {best_overall['feasible']}")

## 9. Wykresy jakości i czasu

Wykresy porównują minimalną wariancję, średnią wariancję, stabilność i czas działania algorytmów.

In [ ]:
def plot_grouped_metric(data: pd.DataFrame, metric: str, ylabel: str, title: str):
    pivot = data.pivot(index="A", columns="algorithm", values=metric)
    ax = pivot.plot(kind="bar", figsize=(9, 4), width=0.8)
    ax.set_title(title)
    ax.set_xlabel("Maksymalna liczba aktywów A")
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    return ax

plot_grouped_metric(summary_results, "best_daily_variance", "Najlepsza dzienna wariancja", "Minimalna wariancja według algorytmu i A")
plot_grouped_metric(summary_results, "mean_daily_variance", "Średnia dzienna wariancja", "Średnia wariancja w powtórzeniach")
plot_grouped_metric(summary_results, "std_objective", "Odchylenie standardowe celu", "Stabilność algorytmów")
plot_grouped_metric(summary_results, "mean_time_sec", "Średni czas [s]", "Czas działania")
plt.show()

## 10. Wykresy dywersyfikacji

Wykresy pokazują efektywną liczbę aktywów, HHI oraz koncentrację największych pozycji.

In [ ]:
plot_grouped_metric(summary_results, "mean_effective_assets", "Efektywna liczba aktywów", "Dywersyfikacja: efektywna liczba aktywów")
plot_grouped_metric(summary_results, "mean_hhi", "HHI", "Koncentracja portfela: Herfindahl-Hirschman Index")
plot_grouped_metric(summary_results, "mean_max_weight", "Średnia największa waga", "Ryzyko koncentracji w jednej pozycji")
plot_grouped_metric(summary_results, "mean_top3_weight", "Udział trzech największych pozycji", "Koncentracja w trzech największych pozycjach")
plt.show()

## 11. Dywersyfikacja a wariancja

Wykres punktowy pokazuje relację między dywersyfikacją portfela a osiągniętą wariancją.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for algorithm, group in results.groupby("algorithm"):
    ax.scatter(group["effective_assets"], group["daily_variance"], label=algorithm, alpha=0.75)
ax.set_title("Dywersyfikacja a osiągnięta wariancja")
ax.set_xlabel("Efektywna liczba aktywów")
ax.set_ylabel("Dzienna wariancja portfela")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 12. Udział klas aktywów

Wykresy skumulowane pokazują, jaką część portfeli stanowią aktywa konserwatywne, zrównoważone i agresywne.

In [ ]:
class_cols = ["mean_share_conservative", "mean_share_balanced", "mean_share_aggressive"]
class_labels = ["konserwatywne", "zrównoważone", "agresywne"]
for max_assets in A_VALUES:
    tmp = summary_results[summary_results["A"] == max_assets].set_index("algorithm")[class_cols]
    ax = tmp.plot(kind="bar", stacked=True, figsize=(8, 4), color=["#4C78A8", "#72B7B2", "#F58518"])
    ax.set_title(f"Udział klas aktywów w portfelach, A={max_assets}")
    ax.set_xlabel("Algorytm")
    ax.set_ylabel("Średni udział w portfelu")
    ax.set_ylim(0, 1)
    ax.legend(class_labels, title="Klasa aktywów", loc="upper right")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 13. Wpływ kardynalności

Wykres pokazuje, jak zmiana limitu aktywów `A` wpływa na najlepszą osiągniętą wariancję.

In [ ]:
plt.figure(figsize=(9, 4))
for algorithm in results["algorithm"].unique():
    tmp = summary_results[summary_results["algorithm"] == algorithm].sort_values("A")
    plt.plot(tmp["A"], tmp["best_daily_variance"], marker="o", label=algorithm)
plt.title("Wpływ ograniczenia kardynalności na minimalną wariancję")
plt.xlabel("Maksymalna liczba aktywów A")
plt.ylabel("Najlepsza dzienna wariancja")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 14. Skład najlepszego portfela

Pokazywane są wagi najlepszego portfela oraz tabela aktywnych pozycji z klasą aktywa.

In [ ]:
best_weights = best_overall["weights"]
active = np.where(best_weights > 1e-8)[0]
colors = {"konserwatywne": "#4C78A8", "zrównoważone": "#72B7B2", "agresywne": "#F58518"}
bar_colors = [colors[asset_class[i]] for i in active]
plt.figure(figsize=(max(8, 0.55 * len(active)), 4))
plt.bar([asset_names[i] for i in active], best_weights[active] * 100, color=bar_colors)
plt.title("Wagi najlepszego portfela według minimalnej wariancji")
plt.xlabel("Aktywo")
plt.ylabel("Waga [%]")
plt.xticks(rotation=45, ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

best_portfolio_table = pd.DataFrame({
    "asset": [asset_names[i] for i in active],
    "class": [asset_class[i] for i in active],
    "weight": best_weights[active],
    "weight_pct": best_weights[active] * 100,
}).sort_values("weight", ascending=False)
best_portfolio_table

## 15. Zbieżność

Wykres przedstawia historię najlepszej wartości funkcji celu dla najlepszego ustawienia z eksperymentu.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for (algorithm, max_assets, run), hist in histories.items():
    if max_assets == int(best_overall["A"]) and run == int(best_overall["run"]):
        ax.plot(hist, label=f"{algorithm}, A={max_assets}, run={run}")
ax.set_title("Zbieżność algorytmów dla najlepszego ustawienia A i uruchomienia")
ax.set_xlabel("Iteracja / generacja")
ax.set_ylabel("Najlepsza dotąd wartość funkcji celu")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 16. Walidacja wyników

Funkcja kontrolna sprawdza sumę wag, nieujemność, kardynalność, minimalny zwrot i wymiar macierzy kowariancji.

In [ ]:
def validate_results(results_df: pd.DataFrame, target_return: float):
    failures = []
    for idx, row in results_df.iterrows():
        weights = row["weights"]
        if not np.isclose(weights.sum(), 1.0, atol=1e-6):
            failures.append((idx, "suma wag różna od 1"))
        if np.any(weights < -1e-8):
            failures.append((idx, "ujemne wagi"))
        if cardinality(weights) > row["A"]:
            failures.append((idx, "przekroczona kardynalność"))
        if portfolio_return(weights, mu) + 1e-8 < target_return:
            failures.append((idx, "niespełniony minimalny zwrot"))
    if cov.shape != (n_assets, n_assets):
        failures.append(("cov", "niepoprawny wymiar macierzy kowariancji"))
    if failures:
        raise AssertionError(f"Wykryto naruszenia ograniczeń: {failures[:10]}")
    return "Wszystkie sprawdzone rozwiązania spełniają ograniczenia."

validate_results(results, TARGET_RETURN)

## 17. Interpretacja automatyczna

Komórka wypisuje najważniejsze wyniki: najlepszą wariancję, najszybszą konfigurację, stabilność i dywersyfikację.

In [ ]:
best_per_algorithm = results.sort_values("daily_variance").groupby("algorithm").head(1).sort_values("daily_variance")
fastest = summary_results.sort_values("mean_time_sec").iloc[0]
most_stable = summary_results.sort_values("std_objective").iloc[0]
most_diversified = summary_results.sort_values("mean_effective_assets", ascending=False).iloc[0]

print("Najlepsze pojedyncze wyniki według minimalnej wariancji:")
for _, row in best_per_algorithm.iterrows():
    print(
        f"- {row['algorithm']}: A={row['A']}, wariancja={row['daily_variance']:.10f}, "
        f"roczny zwrot={row['annual_return']:.4%}, roczna zmienność={row['annual_volatility']:.4%}, "
        f"efektywna liczba aktywów={row['effective_assets']:.2f}, czas={row['time_sec']:.3f}s"
    )

print()
print("Najszybsza konfiguracja średnio:")
print(f"- {fastest['algorithm']} dla A={int(fastest['A'])}, średni czas {fastest['mean_time_sec']:.3f}s")

print()
print("Najstabilniejsza konfiguracja według odchylenia wartości celu:")
print(f"- {most_stable['algorithm']} dla A={int(most_stable['A'])}, std celu {most_stable['std_objective']:.10f}")

print()
print("Najbardziej zdywersyfikowana konfiguracja średnio:")
print(f"- {most_diversified['algorithm']} dla A={int(most_diversified['A'])}, efektywna liczba aktywów {most_diversified['mean_effective_assets']:.2f}")

print()
print("Interpretacja:")
print(
    "Algorytm rozwiązujący zadanie najlepiej to ten, który przy spełnieniu ograniczeń uzyskuje najniższą wariancję. "
    "Dywersyfikację analizujemy równolegle: wyższa efektywna liczba aktywów i niższy HHI oznaczają mniejszą koncentrację portfela. "
    "Jeżeli portfel ma bardzo niską wariancję, ale jest silnie skoncentrowany, warto wskazać ten kompromis w opisie wyników. "
    "Udział klas aktywów pokazuje, czy niska wariancja wynika głównie z wyboru aktywów konserwatywnych, czy z mieszania różnych profili ryzyka."
)